# 1. Quick start

Find and inspect open SAR scenes from **ICEYE**, **Umbra** and **Capella**.

```bash
pip install open-sar-triad
```

No API key, no account. The backend is a static catalog on a CDN, so there is
nothing to authenticate against and nothing to rate limit.

In [ ]:
# Point at the hosted API, or a local build for testing.
#   BASE = 'http://localhost:8000/api/v1'   # after: python3 -m http.server 8000
BASE = None

from opensartriad import Catalog

cat = Catalog(BASE) if BASE else Catalog()
cat

## What is in the catalog?

`stats()` is a small file describing the whole collection. Cheap to call.

In [ ]:
stats = cat.stats()

print('total scenes :', stats['total'])
print('by provider  :', stats['by_provider'])
print('date range   :', stats['temporal_extent'])
print('last built   :', stats['generated'][:10])

In [ ]:
# Sensor modes and formats actually present
print('modes  :', stats['by_mode'])
print('formats:', stats['by_format'])

## A first search

All filters combine with AND. The bbox is `(west, south, east, north)` in EPSG:4326.

The first search downloads a ~0.6 MB index and caches it, so **reuse one `Catalog`**
and later searches are instant and entirely local.

In [ ]:
scenes = cat.search(
    bbox=(5.9, 47.2, 10.5, 55.1),   # roughly Germany
    start='2025-01-01',
    end='2026-01-01',
)
scenes

In [ ]:
# SceneCollection behaves like a list
print(len(scenes))
for s in scenes[:5]:
    print(f'{s.date}  {s.provider:8} {s.mode or "?":10} {s.id[:46]}')

## Anatomy of a scene

These attributes come from the search index and cost nothing.

In [ ]:
s = scenes[0]

print('id       :', s.id)
print('provider :', s.provider)
print('date     :', s.date, '| year:', s.year)
print('mode     :', s.mode)
print('orbit    :', s.orbit, '| look:', s.look)
print('formats  :', s.formats)
print('bbox     :', s.bbox)

## Lazy loading

Download URLs are **not** in the search index, because including them would make it
far larger. They live in the per-provider files, which the client fetches the first
time you ask for one and then caches.

So the next cell is the slow one. Everything after it is fast.

In [ ]:
fmt = s.formats[0]

print('format      :', fmt)
print('download    :', s.url(fmt))
print('sidecar     :', s.metadata_url(fmt))
print('polarization:', s.properties.get('sar:polarizations'))
print('resolution  :', s.properties.get('sar:resolution_range'))

## Licence

Scene metadata is **CC-BY 4.0**. When you publish derived work, credit the
originating provider. Imagery is never proxied by this project: every download URL
points at the provider's own storage.

In [ ]:
lic = cat.license()
print(lic['license'], '-', lic['license_url'])
print()
print(lic['attribution'])

---

Next: **02_search_and_download.ipynb**, which covers narrowing a search and
pulling the actual products down.